# Document Question Answering System (RAG)

A simple Retrieval-Augmented Generation system for answering questions from a PDF document.

## Objective

Build a beginner-friendly RAG pipeline that retrieves useful passages from a custom document before generating an answer. This keeps responses grounded in the uploaded document.

## Import Libraries

Install the small set of libraries used in this notebook. These commands are compatible with Google Colab and only need to run once per session.

In [1]:
!pip -q install -U langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu pypdf sentence-transformers transformers 2>/dev/null

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Libraries are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
Libraries are ready.


## Load Document

Upload one PDF from your computer. `PyPDFLoader` reads its pages and creates LangChain document objects for the retrieval pipeline.

In [2]:
uploaded_files = files.upload()
pdf_name = next((name for name in uploaded_files if name.lower().endswith('.pdf')), None)

if pdf_name is None:
    raise ValueError('Please upload a PDF file.')

documents = PyPDFLoader(pdf_name).load()
print(f'Loaded {len(documents)} page(s) from: {pdf_name}')

Saving Week7_Project.pdf to Week7_Project.pdf
Loaded 4 page(s) from: Week7_Project.pdf


## Split into Chunks

Long pages are split into overlapping chunks. Overlap helps preserve meaning when an important sentence crosses a chunk boundary.

In [3]:
# Normalize extra whitespace before splitting the document.
for document in documents:
    document.page_content = ' '.join(document.page_content.split())

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)
chunks = text_splitter.split_documents(documents)
print(f'Created {len(chunks)} chunks.')

Created 6 chunks.


## Create Embeddings

Embeddings convert text into numerical vectors. Similar meanings are placed closer together, allowing relevant chunks to be found for a question.

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
print('Embedding model loaded.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## Store in Vector Database

FAISS stores the chunk embeddings in memory. It provides fast similarity search without requiring a separate database server.

In [5]:
vector_database = FAISS.from_documents(chunks, embedding_model)
print('FAISS vector database created.')

FAISS vector database created.


## Build Retriever

The retriever selects the three chunks most similar to a user question. These chunks will become the context supplied to the language model.

In [6]:
retriever = vector_database.as_retriever(search_kwargs={'k': 3})
print('Retriever is ready.')

Retriever is ready.


## Build RAG Pipeline

This function retrieves document context and asks a lightweight Hugging Face model to answer only from that context. The model reports when the document does not contain the answer.

In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def answer_question(question):
    """Retrieve relevant text and generate a grounded answer."""
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say: I could not find the answer in the document.

Context:
{context}

Question: {question}
Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    output_ids = model.generate(**inputs, max_new_tokens=120)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return response, retrieved_docs

print("RAG pipeline is ready.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

RAG pipeline is ready.


## Ask Sample Questions

Replace these examples with questions about your uploaded document. Keeping questions specific usually produces clearer retrieved context and answers.

In [8]:
sample_questions = [
    "What is Retrieval-Augmented Generation (RAG)?",
    "What are the main steps in a RAG pipeline?",
    "Why are documents split into chunks?",
    "What is the role of a vector database in RAG?"
]

## Display Answers

Run the pipeline for each sample question and print the generated answer. The page numbers show where the retrieved supporting information came from.

In [9]:
for question in sample_questions:
    answer, source_docs = answer_question(question)
    pages = sorted({doc.metadata.get('page', 0) + 1 for doc in source_docs})
    print(f'Question: {question}')
    print(f'Answer: {answer}')
    print(f'Retrieved page(s): {pages}\n')

Question: What is Retrieval-Augmented Generation (RAG)?
Answer: Answer questions based on custom documents
Retrieved page(s): [1, 3]

Question: What are the main steps in a RAG pipeline?
Answer: Using a RAG pipeline, you can use a RAG pipeline to analyze the data and generate answers.
Retrieved page(s): [1, 3]

Question: Why are documents split into chunks?
Answer: to improve retrieval accuracy
Retrieved page(s): [2, 3]

Question: What is the role of a vector database in RAG?
Answer: a database that can understand user queries, retrieve relevant information, and generate accurate answers
Retrieved page(s): [1, 3]



## Conclusion

A RAG-based question answering system was successfully built.
It retrieves relevant document chunks before generating an answer.
This improves accuracy compared with answering without document context.
Future improvements include better embedding models, hybrid search, and reranking.